## HomeWork 4 NLP
# start

In [ ]:
#Предисловие:
#Вообще у меня тут много приколов
#я постарался почистить откровенный мусор
#но как то не все получилось убрать
# из интересного это прикольная чистка чтобы токены были более полезные
# ака ансамбль из 5 моделек, которые вместе решают какой ответ лучше подходит
# ну еще приколы с тем, что я поворовал уверенные данные из тестовой выборки
# и еще немножко ее дообучал по злому без удовольствия
# и вроде даже нормальный результат получился, можно еще лучше сделать конечно, но понимание куда двигаться пока что нет(
# ну и чтобы по 1000 раз не обучать каждую модель, все сохраняются в папочки для удобства и можно залоудить, а не заново запускать программу

In [1]:
import re
import hashlib
import pandas as pd
import fasttext
import csv
import os

from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[./:_\-?&=]', ' ', text)
    text = text.replace('https', '').replace('http', '').replace('www', '')
    text = re.sub(r'[^a-zA-Zа-яА-Я0-9\s]', ' ', text)
    text = " ".join(text.lower().split())
    
    return text.strip()


def print_results(N, p, r):
    print("Precision\t{:.3f}".format(p))
    print("Recall\t{:.3f}".format(r))
    print("F1\t{:.3f}".format(2*p*r/(p+r)))


import re

def predict_ensemble(url, title, models):
    clean_t = clean_text(f"{url} {title}")
    scores = {}
    model_predictions = []

    for m in models:
        labels, probs = m.predict(clean_t, k=1)
        lbl, pr = labels[0], probs[0]
        scores[lbl] = scores.get(lbl, 0) + pr
        model_predictions.append((lbl, pr))
    
    best_label_str = max(scores, key=scores.get)
    try:
        best_label_id = int(re.search(r'\d+', best_label_str).group())
    except:
        best_label_id = 0
    
    is_c = all(lbl == best_label_str and pr > 0.999 for lbl, pr in model_predictions)
    
    if is_c:
        doptraindata_list.append({
            'url': url,
            'title': title,
            'label': best_label_id
        })

    return best_label_id


In [3]:
dataset = pd.read_csv("./train.csv", names = ['ID','url','title','label'], dtype = {'ID':int,'url':str,'title':str,'label':int},header=0)
doptraindata = pd.DataFrame
doptraindata_list = []
dataset.shape
#doptraindata = pd.DataFrame(doptraindata_list)

(135309, 4)

In [4]:
dataset = dataset[["url", "title","label"]]
print(dataset['label'].value_counts())
dataset.shape

label
0    118594
1     16715
Name: count, dtype: int64


(135309, 3)

Все хорошо, ничего не сломали. Пока что...

In [5]:
configs = [
    {'lr': 0.2, 'epoch': 20, 'wordNgrams': 2, 'dim': 100, 'minn': 3, 'maxn': 6},
    {'lr': 0.1, 'epoch': 25, 'wordNgrams': 3, 'dim': 150, 'minn': 2, 'maxn': 5},
    {'lr': 0.3, 'epoch': 15, 'wordNgrams': 2, 'dim': 200, 'minn': 4, 'maxn': 7},
    {'lr': 0.5, 'epoch': 10, 'wordNgrams': 2, 'dim': 50, 'minn': 3, 'maxn': 5},
    {'lr': 0.3, 'epoch': 20, 'wordNgrams': 4, 'dim': 100, 'minn': 3, 'maxn': 7}
]

In [6]:
def train(dataset, configs, save_dir="saved_models"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    train_dataset = dataset.apply(
        lambda x: f"__label__{x['label']} {clean_text(x['url'])} {clean_text(x['title'])}", axis=1)
    print(train_dataset.shape)
    train_dataset.to_csv('trainft.txt', index=False, header=False, quoting=csv.QUOTE_NONE, escapechar=" ")
    new_models = []
    for i, cfg in enumerate(configs):
        print(f"Обучение модели {i+1}...")
        m = fasttext.train_supervised(input="trainft.txt", **cfg)
        new_models.append(m)
        model_path = os.path.join(save_dir, f"model_{i+1}.bin")
        m.save_model(model_path)
        print(f"Модель сохранена: {model_path}")
    
    return new_models 

In [ ]:
models = train(dataset, configs) 

(135309,)
Обучение модели 1...


In [ ]:
testtask = pd.read_csv("./test.csv", names = ['ID','url','title'], dtype = {'ID':int,'url':str,'title':str},header=0)
len(models)

In [ ]:
def anscreate(testtask, models):
    ans = pd.DataFrame()
    ans['ID'] = testtask['ID']

    ans['label'] = testtask.apply(lambda row: predict_ensemble(row['url'], row['title'], models), axis=1)
    
    ans['label'] = pd.to_numeric(ans['label'], errors='coerce').fillna(0).astype(int)
    
    ans.loc[~ans['label'].isin([0, 1]), 'label'] = 0 
    
    print("Уникальные значения в submission:", ans['label'].unique())
    ans.to_csv('submission.csv', index=False)

In [ ]:
anscreate(testtask,models)

In [ ]:
doptraindata = pd.DataFrame(doptraindata_list)
testtask.shape, doptraindata.shape

In [ ]:
if not doptraindata.empty:
    newdataset = pd.concat([dataset,dataset,doptraindata], ignore_index=True)
    newdataset['label'] = newdataset['label'].astype(int)
print(newdataset['label'].value_counts())

In [ ]:
print(newdataset['label'].value_counts())
newdataset.shape

In [ ]:
models = train(newdataset,configs,save_dir="last_models")

In [ ]:
anscreate(testtask,models)

In [ ]:
#pd.unique(submission["label"])

In [ ]:
def load_models(save_dir="saved_models"):
    loaded_models = []
    if not os.path.exists(save_dir):
        print("Папка с моделями не найдена!")
        return []
    
    files = sorted([f for f in os.listdir(save_dir) if f.endswith('.bin')])
    for f in files:
        path = os.path.join(save_dir, f)
        print(f"Загрузка {path}...")
        m = fasttext.load_model(path)
        loaded_models.append(m)
    
    return loaded_models

In [ ]:
# lm = load_models()
# anscreate(testtask,lm)